In [0]:
create table `data-growth-analytic`.bronze.crm_events as
select * from read_files('/Volumes/data-growth-analytic/bronze/raw/crm_events.csv', format=>'csv', header=>'true',inferColumnTypes=>'false')

In [0]:
create table `data-growth-analytic`.bronze.customers as
select * from read_files('/Volumes/data-growth-analytic/bronze/raw/customers.csv', format=>'csv', header=>'true',inferColumnTypes=>'false')

In [0]:
create table `data-growth-analytic`.bronze.marketing_campaigns as
select * from read_files('/Volumes/data-growth-analytic/bronze/raw/marketing_campaigns.csv', format=>'csv', header=>'true',inferColumnTypes=>'false')

In [0]:
create table `data-growth-analytic`.bronze.marketing_order_items as
select * from read_files('/Volumes/data-growth-analytic/bronze/raw/order_items.csv', format=>'csv', header=>'true',inferColumnTypes=>'false')

In [0]:
create table `data-growth-analytic`.bronze.marketing_orders as
select * from read_files('/Volumes/data-growth-analytic/bronze/raw/orders.csv', format=>'csv', header=>'true',inferColumnTypes=>'false')

In [0]:
create table `data-growth-analytic`.bronze.products as
select * from read_files('/Volumes/data-growth-analytic/bronze/raw/products.csv', format=>'csv', header=>'true',inferColumnTypes=>'false')

In [0]:
Select 'crm_events' as tabela, count(*) as linhas from `data-growth-analytic`.bronze.crm_events
Union all select 'products', count(*) from `data-growth-analytic`.bronze.products
Union all select 'customers', count(*) from `data-growth-analytic`.bronze.customers
Union all select 'marketing_campaigns', count(*) from `data-growth-analytic`.bronze.marketing_campaigns
Union all select 'marketing_orders', count(*) from `data-growth-analytic`.bronze.marketing_orders
Union all select 'marketing_order_items', count(*) from `data-growth-analytic`.bronze.marketing_order_items

In [0]:
SELECT
    COUNT(*) AS total_eventos,
    COUNT(campaign_name) AS campanhas_preenchidas,
    COUNT(DISTINCT campaign_name) AS campanhas_distintas
FROM `data-growth-analytic`.bronze.crm_events;

In [0]:
SELECT
    COUNT(*) AS total_campanhas,
    COUNT(campaign_name) AS nomes_preenchidos,
    COUNT(DISTINCT campaign_name) AS campanhas_distintas
FROM `data-growth-analytic`.bronze.marketing_campaigns;

In [0]:
SELECT
    ce.campaign_name AS evento,
    camp.campaign_name AS campanha,
    camp.channel AS canal
FROM `data-growth-analytic`.bronze.crm_events ce

LEFT JOIN `data-growth-analytic`.bronze.marketing_campaigns camp
    ON TRIM(LOWER(ce.campaign_name)) =
       TRIM(LOWER(camp.campaign_name))

LIMIT 20;

In [0]:
DESCRIBE `data-growth-analytic`.bronze.crm_events;

In [0]:
SELECT DISTINCT
    ce.campaign_name AS campanha_evento,
    ce.channel AS canal_evento,
    camp.campaign_name AS campanha_marketing,
    camp.channel AS canal_marketing

FROM `data-growth-analytic`.bronze.crm_events ce

LEFT JOIN `data-growth-analytic`.bronze.marketing_campaigns camp
    ON LOWER(TRIM(ce.campaign_name)) =
       LOWER(TRIM(camp.campaign_name))

LIMIT 30;

In [0]:
select * from `data-growth-analytic`.bronze.rascunho limit 10

In [0]:
CREATE OR REPLACE VIEW
`data-growth-analytic`.bronze.rascunho AS

SELECT
    -- Customer
    c.customer_id,
    c.signup_date,
    c.acquisition_channel,
    c.region,
    c.age_band,
    c.customer_segment,

    -- Order
    o.order_id,
    o.order_date,
    o.order_status,
    o.payment_method,

    -- Order items
    oi.product_id,
    oi.quantity,
    oi.unit_price,
    oi.discount_pct,

    CAST(oi.quantity AS DOUBLE) *
    CAST(oi.unit_price AS DOUBLE) AS gross_item_revenue,

    -- Campaign
    ce.campaign_name,
    ce.channel AS campaign_channel,

    -- Campaign events
    ce.sent,
    ce.opened,
    ce.clicked,
    ce.converted,
    ce.sent_date

FROM `data-growth-analytic`.bronze.customers c

LEFT JOIN `data-growth-analytic`.bronze.marketing_orders o
    ON c.customer_id = o.customer_id

LEFT JOIN `data-growth-analytic`.bronze.marketing_order_items oi
    ON o.order_id = oi.order_id

LEFT JOIN `data-growth-analytic`.bronze.crm_events ce
    ON c.customer_id = ce.customer_id;

In [0]:
SELECT
    campaign_name,
    campaign_channel,
    COUNT(*) AS registros,
    SUM(converted) AS conversoes

FROM `data-growth-analytic`.bronze.rascunho

GROUP BY
    campaign_name,
    campaign_channel

ORDER BY conversoes DESC;